# Tratamento de Dados com Python
**Instituição:** IPASI - Instituto de Pesquisas Avançadas e Soluções Inovadoras


Nesta aula prática, vamos simular o recebimento de uma base de dados real de pacientes/usuários. Nosso objetivo é transformar dados brutos e inconsistentes em um *dataset* íntegro, pronto para alimentar modelos estatísticos e algoritmos de Inteligência Artificial.

In [6]:
#%pip install pandas
import pandas as pd

df = pd.read_csv('dados_pacientes.csv')
display(df.head(5))

,id_usuario,nome,idade,freq_cardiaca_repouso,data_cadastro,mensalidade_plano
0,1,Ana Silva,28.0,68.0,15/01/2026,R$ 150.00
1,2,Carlos Santos,35.0,72.0,16/01/2026,R$ 200.50
2,3,Beatriz Lima,NaN,65.0,16/01/2026,R$ 150.00
3,4,Daniel Souza,42.0,80.0,17/01/2026,R$ 300.00
4,5,Eduarda Costa,25.0,62.0,18/01/2026,R$ 150.00


## Diagnóstico: Onde estão os problemas?

In [7]:
print("--- Tipos de Dados ---")
df.info()

print("\n--- Contagem de Valores Nulos (NaN) ---")
print(df.isnull().sum())

--- Tipos de Dados ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_usuario             8 non-null      int64  
 1   nome                   8 non-null      object 
 2   idade                  6 non-null      float64
 3   freq_cardiaca_repouso  7 non-null      float64
 4   data_cadastro          8 non-null      object 
 5   mensalidade_plano      8 non-null      object 
dtypes: float64(2), int64(1), object(3)
memory usage: 516.0+ bytes

--- Contagem de Valores Nulos (NaN) ---
id_usuario               0
nome                     0
idade                    2
freq_cardiaca_repouso    1
data_cadastro            0
mensalidade_plano        0
dtype: int64


## Tratando Duplicatas

In [8]:
df_limpo = df.drop_duplicates().copy()

print(f"Registros antes da limpeza: {len(df)}")
print(f"Registros após a limpeza: {len(df_limpo)}")

Registros antes da limpeza: 8
Registros após a limpeza: 7


## Imputação: Lidando com a Valores Ausentes

In [9]:
mediana_idade = df_limpo['idade'].median()
df_limpo['idade'] = df_limpo['idade'].fillna(mediana_idade)

media_freq = round(df_limpo['freq_cardiaca_repouso'].mean(), 1)
df_limpo['freq_cardiaca_repouso'] = df_limpo['freq_cardiaca_repouso'].fillna(media_freq)

print("Valores nulos após imputação:")
print(df_limpo[['idade', 'freq_cardiaca_repouso']].isnull().sum())

Valores nulos após imputação:
idade                    0
freq_cardiaca_repouso    0
dtype: int64


## Transformação de Tipos (Casting)

In [10]:
df_limpo['data_cadastro'] = pd.to_datetime(df_limpo['data_cadastro'], format='%d/%m/%Y')

df_limpo['mensalidade_plano'] = df_limpo['mensalidade_plano'].str.replace('R\$ ', '', regex=True).astype(float)

df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_usuario             7 non-null      int64         
 1   nome                   7 non-null      object        
 2   idade                  7 non-null      float64       
 3   freq_cardiaca_repouso  7 non-null      float64       
 4   data_cadastro          7 non-null      datetime64[ns]
 5   mensalidade_plano      7 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 392.0+ bytes


## Feature Engineering

In [12]:

data_atual = pd.Timestamp.today().normalize() # Data atual desoncisderando as horas
df_limpo['dias_com_plano'] = (data_atual - df_limpo['data_cadastro']).dt.days
df_limpo.head()

,id_usuario,nome,idade,freq_cardiaca_repouso,data_cadastro,mensalidade_plano,dias_com_plano
0,1,Ana Silva,28.0,68.0,2026-01-15,150.0,46
1,2,Carlos Santos,35.0,72.0,2026-01-16,200.5,45
2,3,Beatriz Lima,30.0,65.0,2026-01-16,150.0,45
3,4,Daniel Souza,42.0,80.0,2026-01-17,300.0,44
4,5,Eduarda Costa,25.0,62.0,2026-01-18,150.0,43


In [13]:
display(df_limpo.describe())

,id_usuario,idade,freq_cardiaca_repouso,data_cadastro,mensalidade_plano,dias_com_plano
count,7.000000,7.000000,7.000000,7,7.000000,7.000000
mean,4.000000,31.428571,69.500000,2026-01-17 06:51:25.714285824,185.857143,43.714286
min,1.000000,25.000000,62.000000,2026-01-15 00:00:00,150.000000,41.000000
25%,2.500000,29.000000,66.500000,2026-01-16 00:00:00,150.000000,42.500000
50%,4.000000,30.000000,69.500000,2026-01-17 00:00:00,150.000000,44.000000
75%,5.500000,32.500000,71.000000,2026-01-18 12:00:00,200.500000,45.000000
max,7.000000,42.000000,80.000000,2026-01-20 00:00:00,300.000000,46.000000
std,2.160247,5.533448,5.708181,NaN,55.678178,1.799471
